In [1]:
import pandas as pd
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# Load both datasets
df = pd.read_csv('path to Model generations csv')  # Dataset 1: Model predictions
df2 = pd.read_csv('path to ground truth csv')  # Dataset 2: Ground truth labels (majority labels)

In [2]:
# Define model columns and final annotated labels column
model_columns = ['Manipulation Type']
final_labels_column = 'Majority_Labels'

# Define your 12 classes
class_labels = ['N_A', 'DEN', 'EVA', 'FEI', 'RAT', 'VIC', 'SER', 'S_B', 'INT', 'B_A', 'ACC', 'P_S']

# Convert the final labels column into a list of labels, handling NaN values and cleaning the text
df2[final_labels_column] = df2[final_labels_column].apply(lambda x: [label.strip() for label in x.split(',')] if pd.notna(x) else [])

# Clean the model predictions in Dataset 1 as well
for model_column in model_columns:
    df[model_column] = df[model_column].apply(
    lambda x: [label.strip() for label in x.replace('No.', 'N_A').replace('Answer: ', '').split(',')] if pd.notna(x) else []
)



In [3]:
# Initialize MultiLabelBinarizer with the defined 12 classes
mlb = MultiLabelBinarizer(classes=class_labels)

# Binarize the true labels from Dataset 2 (Majority Labels)
y_true = mlb.fit_transform(df2[final_labels_column])
print(mlb.classes_)

['N/A' 'DEN' 'EVA' 'FEI' 'RAT' 'VIC' 'SER' 'S_B' 'INT' 'B_A' 'ACC' 'P_S']


In [4]:
# Initialize a dictionary to store the results for each model
results = {}

# Loop over each model column in Dataset 1 and calculate metrics
for model_column in model_columns:
    y_pred = mlb.transform(df[model_column])
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=1)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=1)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=1)
    # Store the results for this model and prompting method
    results[model_column] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

# Print the results for each model and prompting method
for model, metrics in results.items():
    print(f"Results for {model}:")
    print(f"  Accuracy: {metrics['Accuracy']:.4f}")
    print(f"  Precision: {metrics['Precision']}")
    print(f"  Recall: {metrics['Recall']}")
    print(f"  F1 Score: {metrics['F1 Score']}")
    print()

Results for Manipulation Type:
  Accuracy: 0.0000
  Precision (Class-wise): 0.15385306027803228
  Recall (Class-wise): 0.19114357864357864
  F1 Score (Class-wise): 0.2857535196240885



In [5]:
import numpy as np
def calculate_classwise_accuracy(y_true, y_pred):
    # Convert y_true and y_pred to NumPy arrays if they are not already
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = y_true.shape[1]
    
    # Initialize list to store class-wise accuracy
    classwise_accuracies = []
    
    for i in range(num_classes):
        # Calculate accuracy for each class
        correct_predictions = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 1))
        total_true_labels = np.sum(y_true[:, i] == 1)
        
        if total_true_labels > 0:
            accuracy = correct_predictions / total_true_labels
        else:
            accuracy = 0.0  # No instances of this label in true labels
        
        classwise_accuracies.append(accuracy)
    
    # Calculate the mean accuracy
    mean_accuracy = np.mean(classwise_accuracies)
    
    return classwise_accuracies, mean_accuracy

classwise_accuracies, mean_accuracy = calculate_classwise_accuracy(y_true, y_pred)

classwise_accuracies, mean_accuracy 

([0.75,
  0.16666666666666666,
  0.0,
  0.1,
  0.42857142857142855,
  0.0,
  0.1,
  0.2,
  0.18181818181818182,
  0.1,
  0.1,
  0.16666666666666666],
 0.19114357864357864)